In [25]:
import pandas as pd

In [26]:
happiness_df = pd.read_csv("happiness.csv")
steps_df = pd.read_csv("world_map_steps_average.csv")
wine_df = pd.read_csv("wine-consumption-per-capita.csv")
diet_df = pd.read_csv("dietary-composition-by-country.csv")
life_expectancy_df = pd.read_csv("life-expectancy.csv")

### Individual features

##### 1. Happiness (no ISO code)

In [27]:
happiness_df = happiness_df.rename(columns={"Life evaluation (3-year average)": "Life evaluation", "Country name": "country"})
mean_happiness_by_country = happiness_df.groupby(by="country")["Life evaluation"].mean().reset_index()

##### 2. Steps (no ISO code)

In [28]:
steps_df = steps_df.rename(columns={"region": "country"})

##### 3. Wine consumption

In [29]:
wine_df = wine_df.rename(
    columns={
        "Alcohol, recorded per capita (15+) consumption (in litres of pure alcohol) - Beverage types: Wine": "Wine Consumption", 
        "Code": "ISO",
        "Entity": "country"
        }
    )
mean_wine_consumption_by_country = wine_df.groupby(by=["ISO", "country"])["Wine Consumption"].mean().reset_index()

##### 4. Diet

In [30]:
diet_df = diet_df.rename(columns={"Entity": "country", "Code": "ISO"})
diet_df = diet_df[["country", "ISO", "Pulses"]]
mean_pulses_consumption_by_country = diet_df.groupby(by=["ISO", "country"])["Pulses"].mean().reset_index()

##### 5. Life expectancy

In [31]:
life_expectancy_df = life_expectancy_df.rename(columns={"Entity": "country", "Code": "ISO"})
mean_life_expectancy_by_country = life_expectancy_df.groupby(by=["ISO", "country"])["Life expectancy"].mean().reset_index()

### Blue Zone Index

In [32]:
merged_stats = \
    mean_life_expectancy_by_country[["ISO", "Life expectancy"]].merge(right=mean_pulses_consumption_by_country, on="ISO", how="outer")\
    .merge(right=mean_wine_consumption_by_country[["ISO", "Wine Consumption"]], on="ISO", how="outer")\
    .merge(right=steps_df, on="country", how="outer")\
    .merge(right=mean_happiness_by_country, on="country", how="outer")

#merged_stats.head()

In [33]:
merged_stats["Pulses"] = merged_stats["Pulses"] / merged_stats["Pulses"].max()
merged_stats["Wine Consumption"] = merged_stats["Wine Consumption"] / merged_stats["Wine Consumption"].max()
merged_stats["steps_mean_filtered"] = merged_stats["steps_mean_filtered"] / merged_stats["steps_mean_filtered"].max()
merged_stats["Life evaluation"] = merged_stats["Life evaluation"] / merged_stats["Life evaluation"].max()

#merged_stats.head()

In [34]:
merged_stats = merged_stats.dropna(subset="country")
merged_stats = merged_stats.fillna(0)

In [35]:
def compute_blue_zone_index(row):
    index = (row["Life evaluation"] + row["steps_mean_filtered"] + row["Wine Consumption"] + row["Pulses"]) / 4
    return index

merged_stats["blue_zone_index"] = merged_stats.apply(lambda row: compute_blue_zone_index(row), axis=1)

In [36]:
merged_stats = merged_stats.set_index("country")
merged_stats.to_json("blue-zone-index.csv", orient="index")
merged_stats.reset_index().to_json("blue-zone-index-scatter-plot.csv", orient="records")